# Solution ③ - Foundry Diagnostic Settings 獨立驗證 Notebook

這個 notebook **只用來驗證方案 ③**（Foundry resource → 專屬 LAW）。
與 `test-logging.ipynb`（方案 ①）完全獨立，不互相干擾。

## 前置作業
1. 已執行 `scripts\deploy-foundry-diag.ps1` 部署方案 ③ 基礎設施
2. 取得專屬 LAW 名稱（output `foundryLawName`）

## 測試流程
- **TC-07a**：Non-streaming chat → 必有完整 token usage
- **TC-07b**：Streaming chat（不帶 include_usage）→ 預期 token 為 null
- **TC-07c**：Streaming chat（帶 `stream_options.include_usage=true`）→ 驗證 token 是否回傳
- **TC-07d**：用 KQL 對照三筆請求在方案 ③ LAW 中的記錄

In [ ]:
# === 設定 ===
import os, time, uuid, json, getpass
from openai import OpenAI

APIM_ENDPOINT     = 'https://testaigw01.azure-api.net'
APIM_API_PATH     = '/kunlenewfoundry01/openai/v1/'   # 同方案 ① 用的 API path
DEPLOYMENT_NAME   = 'Kimi-K2.5'
API_VERSION       = '2024-10-21'
FOUNDRY_LAW_NAME  = 'log-foundry-diag-wp74xfiq7okba'  # ← 已從 deploy-foundry-diag.ps1 輸出填入

if 'APIM_KEY' not in os.environ:
    os.environ['APIM_KEY'] = getpass.getpass('APIM Subscription Key: ')

client = OpenAI(
    base_url=f'{APIM_ENDPOINT}{APIM_API_PATH}',
    api_key=os.environ['APIM_KEY'],
    default_headers={'api-key': os.environ['APIM_KEY']},
)

# 在每個請求加上唯一 marker，方便 KQL 撈
RUN_ID = f'tc07-{int(time.time())}'
print(f'RUN_ID = {RUN_ID}')

## 🔍 驗證用輔助函式（用 az CLI 直接查方案 ③ LAW，不用切 Portal）

In [ ]:
# === 驗證輔助：直接用 az CLI 跑 KQL 查方案 ③ LAW ===
import subprocess, json as _json, time

# 從 deploy-foundry-diag.ps1 輸出的 foundryLawCustomerId
WORKSPACE_ID = '918459f8-e940-4f18-b6fd-394f3c599cca'

def run_kql(query, wait_sec=0):
    if wait_sec:
        print(f'⏳ 等 {wait_sec}s 讓 diagnostic ingest...')
        time.sleep(wait_sec)
    cmd = ['az','monitor','log-analytics','query','-w',WORKSPACE_ID,
           '--analytics-query', query, '-o','json']
    r = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
    if r.returncode != 0:
        print('❌', r.stderr); return []
    return _json.loads(r.stdout) if r.stdout.strip() else []

def show_rows(rows, cols=None):
    if not rows:
        print('  (no rows yet — diagnostic 需 2-5 分鐘 ingest，可稍後再跑這個 cell)')
        return
    cols = cols or list(rows[0].keys())
    print('  ' + ' | '.join(cols))
    print('  ' + '-' * 100)
    for r in rows[:10]:
        print('  ' + ' | '.join(str(r.get(c,''))[:30] for c in cols))

## TC-07a — Non-streaming（基線：必有 token usage）

In [ ]:
marker_a = f'{RUN_ID}-a-nonstream'
resp = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{'role':'user','content': f'[{marker_a}] 用一句話說明什麼是雲原生'}],
    max_tokens=2000,
    stream=False,
)
print('Marker:', marker_a)
print('Usage :', resp.usage)
print('Content head:', (resp.choices[0].message.content or '')[:120])

### 🔍 驗證 TC-07a

**重要修正**：Foundry RequestResponse 的 `properties_s` 只有 metadata（`apiName`/`requestLength`/`responseLength`），**不含 token 細節**。

Token 要查 `AzureMetrics`（per-deployment 聚合）。

In [ ]:
# === 驗證 TC-07a (non-stream) ===
# ⚠️ 重要發現：Foundry RequestResponse 的 properties_s 只有 metadata，
#    apiName / requestLength / responseLength / requestTime / responseTime，
#    「沒有」 prompt_tokens / completion_tokens 細項。
#    Token 要查 AzureMetrics 表（聚合指標 InputTokens / OutputTokens / TotalTokens）
#    只能看到 per-deployment 的總量，無法關聯到單一 marker

# === Step 1: 查這筆請求的 metadata（確認有 ingest）===
q_meta = f'''
AzureDiagnostics
| where TimeGenerated > ago(15m)
| where Category == "RequestResponse"
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| extend P = parse_json(properties_s)
| project TimeGenerated, ResultSignature, DurationMs, OperationName,
          ApiName        = tostring(P.apiName),
          RequestLength  = toint(P.requestLength),
          ResponseLength = toint(P.responseLength)
| order by TimeGenerated desc
| take 5
'''
print(f'查 marker = {marker_a} 的 metadata（這個記錄絕對有）:')
rows_meta = run_kql(q_meta, wait_sec=120)
show_rows(rows_meta, ['TimeGenerated','ResultSignature','DurationMs','ApiName','RequestLength','ResponseLength'])

# === Step 2: 查該時間段的 token metrics（per-deployment 聚合）===
q_tok = f'''
AzureMetrics
| where TimeGenerated > ago(10m)
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where MetricName in ("InputTokens","OutputTokens","TotalTokens")
| extend ModelDeployment = tostring(parse_json(tostring(parse_json(tostring(parse_json(SeriesId)[0])))) )
| summarize Sum = sum(Total) by MetricName, bin(TimeGenerated, 1m)
| order by TimeGenerated desc
'''
print(f'\n該時間 deployment Kimi-K2.5 的 token 總量:')
rows_tok = run_kql(q_tok)
show_rows(rows_tok, ['TimeGenerated','MetricName','Sum'])

# === Step 3: Streaming 示意驗證 ===
print()
print('=' * 60)
print('重點結論：')
print('  • RequestResponse log 只有 metadata（requestLength/responseLength）')
print('  • Token 細節在 AzureMetrics（聚合），無法跟單筆請求關聯')
print('  • 要拿 per-request token + 內容，仍需方案 ① 的 APIM body logging')

## TC-07b — Streaming **不帶** include_usage（預期 token 缺失）

In [ ]:
marker_b = f'{RUN_ID}-b-stream-no-usage'
stream = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{'role':'user','content': f'[{marker_b}] 列三個 Azure 服務'}],
    max_tokens=2000,
    stream=True,
)
last_usage = 'NULL (預期)'
for chunk in stream:
    if chunk.usage is not None:
        last_usage = chunk.usage
print('Marker:', marker_b)
print('Last chunk usage:', last_usage)

### 🔍 驗證 TC-07b

Streaming 請求也一樣只有 metadata。並且 `responseLength` 在 SSE chunked 下可能不準。

In [ ]:
# === 驗證 TC-07b (stream, no include_usage) ===
# ⚠️ 重要發現：Foundry RequestResponse 的 properties_s 只有 metadata，
#    apiName / requestLength / responseLength / requestTime / responseTime，
#    「沒有」 prompt_tokens / completion_tokens 細項。
#    Token 要查 AzureMetrics 表（聚合指標 InputTokens / OutputTokens / TotalTokens）
#    只能看到 per-deployment 的總量，無法關聯到單一 marker

# === Step 1: 查這筆請求的 metadata（確認有 ingest）===
q_meta = f'''
AzureDiagnostics
| where TimeGenerated > ago(15m)
| where Category == "RequestResponse"
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| extend P = parse_json(properties_s)
| project TimeGenerated, ResultSignature, DurationMs, OperationName,
          ApiName        = tostring(P.apiName),
          RequestLength  = toint(P.requestLength),
          ResponseLength = toint(P.responseLength)
| order by TimeGenerated desc
| take 5
'''
print(f'查 marker = {marker_b} 的 metadata（這個記錄絕對有）:')
rows_meta = run_kql(q_meta, wait_sec=120)
show_rows(rows_meta, ['TimeGenerated','ResultSignature','DurationMs','ApiName','RequestLength','ResponseLength'])

# === Step 2: 查該時間段的 token metrics（per-deployment 聚合）===
q_tok = f'''
AzureMetrics
| where TimeGenerated > ago(10m)
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where MetricName in ("InputTokens","OutputTokens","TotalTokens")
| extend ModelDeployment = tostring(parse_json(tostring(parse_json(tostring(parse_json(SeriesId)[0])))) )
| summarize Sum = sum(Total) by MetricName, bin(TimeGenerated, 1m)
| order by TimeGenerated desc
'''
print(f'\n該時間 deployment Kimi-K2.5 的 token 總量:')
rows_tok = run_kql(q_tok)
show_rows(rows_tok, ['TimeGenerated','MetricName','Sum'])

# === Step 3: Streaming 示意驗證 ===
print()
print('=' * 60)
print('重點結論：')
print('  • RequestResponse log 只有 metadata（requestLength/responseLength）')
print('  • Token 細節在 AzureMetrics（聚合），無法跟單筆請求關聯')
print('  • 要拿 per-request token + 內容，仍需方案 ① 的 APIM body logging')

## TC-07c — Streaming **帶** `stream_options.include_usage=True`
驗證 Kimi-K2.5 是否支援 OpenAI 的 `include_usage` 約定。
若支援 → 最後一個 chunk 會帶完整 usage → 方案 ③ 也會記到。

In [ ]:
marker_c = f'{RUN_ID}-c-stream-with-usage'
stream = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{'role':'user','content': f'[{marker_c}] 用一句話介紹 APIM'}],
    max_tokens=2000,
    stream=True,
    stream_options={'include_usage': True},
)
last_usage = None
chunks_seen = 0
for chunk in stream:
    chunks_seen += 1
    if chunk.usage is not None:
        last_usage = chunk.usage
print('Marker:', marker_c)
print('Total chunks:', chunks_seen)
print('Final usage :', last_usage)
if last_usage is None:
    print('⚠️  Kimi-K2.5 不支援 include_usage → 方案 ③ 對 streaming 將拿不到 token')
else:
    print('✅  Kimi-K2.5 支援 include_usage → 方案 ③ 可記到 streaming token')

### 🔍 驗證 TC-07c

`include_usage=True` 只影響 client 端拿到 SDK 的 `usage` 物件，**對 Foundry diagnostic log 無差異** — 都是只有 metadata。

In [ ]:
# === 驗證 TC-07c (stream, include_usage) ===
# ⚠️ 重要發現：Foundry RequestResponse 的 properties_s 只有 metadata，
#    apiName / requestLength / responseLength / requestTime / responseTime，
#    「沒有」 prompt_tokens / completion_tokens 細項。
#    Token 要查 AzureMetrics 表（聚合指標 InputTokens / OutputTokens / TotalTokens）
#    只能看到 per-deployment 的總量，無法關聯到單一 marker

# === Step 1: 查這筆請求的 metadata（確認有 ingest）===
q_meta = f'''
AzureDiagnostics
| where TimeGenerated > ago(15m)
| where Category == "RequestResponse"
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| extend P = parse_json(properties_s)
| project TimeGenerated, ResultSignature, DurationMs, OperationName,
          ApiName        = tostring(P.apiName),
          RequestLength  = toint(P.requestLength),
          ResponseLength = toint(P.responseLength)
| order by TimeGenerated desc
| take 5
'''
print(f'查 marker = {marker_c} 的 metadata（這個記錄絕對有）:')
rows_meta = run_kql(q_meta, wait_sec=120)
show_rows(rows_meta, ['TimeGenerated','ResultSignature','DurationMs','ApiName','RequestLength','ResponseLength'])

# === Step 2: 查該時間段的 token metrics（per-deployment 聚合）===
q_tok = f'''
AzureMetrics
| where TimeGenerated > ago(10m)
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where MetricName in ("InputTokens","OutputTokens","TotalTokens")
| extend ModelDeployment = tostring(parse_json(tostring(parse_json(tostring(parse_json(SeriesId)[0])))) )
| summarize Sum = sum(Total) by MetricName, bin(TimeGenerated, 1m)
| order by TimeGenerated desc
'''
print(f'\n該時間 deployment Kimi-K2.5 的 token 總量:')
rows_tok = run_kql(q_tok)
show_rows(rows_tok, ['TimeGenerated','MetricName','Sum'])

# === Step 3: Streaming 示意驗證 ===
print()
print('=' * 60)
print('重點結論：')
print('  • RequestResponse log 只有 metadata（requestLength/responseLength）')
print('  • Token 細節在 AzureMetrics（聚合），無法跟單筆請求關聯')
print('  • 要拿 per-request token + 內容，仍需方案 ① 的 APIM body logging')

## TC-07d — 等 5-10 分鐘後在【方案 ③ LAW】查詢（修正版）

**重要**：根據 [Microsoft 官方文件](https://learn.microsoft.com/azure/azure-monitor/reference/supported-logs/microsoft-cognitiveservices-accounts-logs)，`RequestResponse` log 的 `properties_s` 只有 metadata，**不含 token 細節**。Token 必須查 `AzureMetrics`。

**Step 1：在方案 ③ LAW 看 metadata（無 token）**：
1. Azure Portal → Log Analytics workspaces → 選 `log-foundry-diag-*` → Logs

```kusto
AzureDiagnostics
| where TimeGenerated > ago(30m)
| where Category == "RequestResponse"
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| extend P = parse_json(properties_s)
| project TimeGenerated, OperationName, ResultSignature, DurationMs,
          ApiName        = tostring(P.apiName),
          RequestLength  = toint(P.requestLength),
          ResponseLength = toint(P.responseLength)
| order by TimeGenerated desc
```

**Step 2：查同段時間的 token 總量**（同一個 LAW，但查 `AzureMetrics` 表）：
```kusto
AzureMetrics
| where TimeGenerated > ago(30m)
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where MetricName in ("InputTokens","OutputTokens","TotalTokens")
| summarize Sum=sum(Total) by MetricName, bin(TimeGenerated, 1m)
| render timechart
```

**預期結果（修正後）**：
| 觀察點 | RequestResponse log | AzureMetrics |
|---|---|---|
| TC-07a/b/c 三筆都看得到 | ✅ 三筆 metadata | ⚠️ 看到該分鐘的 token 總量（無法區分是哪一筆） |
| TotalTokens 欄位 | ❌ **不存在** | ✅ 在 metric 內 |

## 並排比較方案 ① vs ③
切到方案 ① LAW（`log-aigw-*`），對同一段時間跑：
```kusto
AppDependencies
| where TimeGenerated > ago(30m)
| where Name has "chat/completions"
| extend P = parse_json(tostring(Properties))
| project TimeGenerated, ResultCode, DurationMs,
          ReqSize  = strlen(tostring(P["Request-Body"])),
          RespSize = strlen(tostring(P["Response-Body"])),
          Truncated = strlen(tostring(P["Response-Body"])) >= 262000
| order by TimeGenerated desc
```

對比兩邊筆數、時間、status code 是否一致 → 確認雙軌平行運作。